<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/pj2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import re
import html
import urllib.parse

train = pd.read_csv('/content/drive/MyDrive/ESAA/project/project2/train.csv')
test = pd.read_csv('/content/drive/MyDrive/ESAA/project/project2/test.csv')

In [ ]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [ ]:
print("[train shape]", train.shape)
print("[test  shape]", test.shape)
print("[cols]", list(train.columns))

[train shape] (7613, 5)
[test  shape] (3263, 4)
[cols] ['id', 'keyword', 'location', 'text', 'target']


In [ ]:
# Target 분포 확인
print("\n[Target 분포 - 개수]")
print(train["target"].value_counts().rename_axis("target").to_frame("count"))

print("\n[Target 분포 - 비율]")
print(train["target"].value_counts(normalize=True).rename_axis("target").to_frame("ratio").round(5))


[Target 분포 - 개수]
        count
target       
0        4342
1        3271

[Target 분포 - 비율]
          ratio
target         
0       0.57034
1       0.42966


실제 재난과 관련 없는 tweet: 0  
실제 재난과 관련 있는 tweet: 1

## **전처리**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

전체 결측치 확인 및 처리

In [ ]:
# 전체 결측치 확인
missing_summary = pd.concat(
    [train.isnull().sum().rename('train_missing'),
     test.isnull().sum().rename('test_missing')],
    axis=1
)

missing_summary

,train_missing,test_missing
id,0,0.0
keyword,61,26.0
location,2533,1105.0
text,0,0.0
target,0,NaN


In [ ]:
# keyword, location 결측치를 unknown으로 대체
for df in [train, test]:
    df['keyword'] = df['keyword'].fillna('unknown')
    df['location'] = df['location'].fillna('unknown')

In [ ]:
# 결측치 처리 후 다시 확인
missing_summary_after = pd.concat(
    [train.isnull().sum().rename('train_missing_after'),
     test.isnull().sum().rename('test_missing_after')],
    axis=1
)

missing_summary_after

,train_missing_after,test_missing_after
id,0,0.0
keyword,0,0.0
location,0,0.0
text,0,0.0
target,0,NaN


URL, 멘션, 해시태그 처리 함수 정의

`http://t.co/...` 형태의 URL과 `@username` 형태의 멘션이 자주 등장  
이 정보는 그대로 두면 노이즈가 될 수 있지만, 실제 재난 트윗은 뉴스 링크를 포함하는 경우가 많기 때문에 URL의 존재 여부 자체는 힌트가 될 수 있다.   
--> URL과 멘션을 완전히 삭제하지 않고 각각 `urltoken`, `usertoken`으로 치환

해시태그는 `#earthquake`처럼 핵심 키워드를 포함하는 경우가 많으므로 `#` 기호만 제거하고 단어는 보존

In [ ]:
# 정규표현식 패턴 정의
url_pattern = r'https?://\S+|www\.\S+'
mention_pattern = r'@\w+'
hashtag_pattern = r'#(\w+)'
special_pattern = r'[^a-zA-Z0-9\s]'
space_pattern = r'\s+'

# 축약형 일부 처리
contractions = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "i'm": "i am",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "i've": "i have",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "i'll": "i will"
}

def expand_contractions(text):
    for old, new in contractions.items():
        text = text.replace(old, new)
    return text

def clean_tweet_basic(text):
    """
    트윗 원문을 기본 정제하는 함수
    - HTML escape 처리
    - 소문자 변환
    - URL, mention을 대표 토큰으로 치환
    - hashtag는 #만 제거하고 단어는 보존
    - 특수문자, 이모지, 불필요한 공백 제거
    """
    text = html.unescape(str(text))
    text = text.lower()
    text = expand_contractions(text)

    # URL과 멘션은 삭제하지 않고 대표 토큰으로 치환
    text = re.sub(url_pattern, ' urltoken ', text)
    text = re.sub(mention_pattern, ' usertoken ', text)

    # 해시태그는 #만 제거하고 단어는 보존
    text = re.sub(hashtag_pattern, r'\1', text)

    # 이모지 및 깨진 문자 제거
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 특수문자 제거, 숫자는 유지
    text = re.sub(special_pattern, ' ', text)

    # 공백 정리
    text = re.sub(space_pattern, ' ', text).strip()

    return text

URL/멘션/해시태그 존재 여부 변수 생성

URL과 멘션은 텍스트 안에서 `urltoken`, `usertoken`으로 보존, 추가로 존재 여부를 0/1 변수로도 생성  
-->이후 모델링 단계에서 보조 feature로 사용 가능

In [ ]:
for df in [train, test]:
    df['has_url'] = df['text'].str.contains(url_pattern, case=False, regex=True, na=False).astype(int)
    df['has_mention'] = df['text'].str.contains(mention_pattern, regex=True, na=False).astype(int)
    df['has_hashtag'] = df['text'].str.contains(r'#\w+', regex=True, na=False).astype(int)

기본 텍스트 정제 적용

`text_clean`은 URL과 멘션을 대표 토큰으로 바꾸고, 해시태그 단어는 보존한 기본 정제 텍스트

In [ ]:
train['text_clean'] = train['text'].apply(clean_tweet_basic)
test['text_clean'] = test['text'].apply(clean_tweet_basic)

train[['text', 'text_clean', 'has_url', 'has_mention', 'has_hashtag']].head()

,text,text_clean,has_url,has_mention,has_hashtag
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,0,0,1
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,0,0,0
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,0,0,0
3,"13,000 people receive #wildfires evacuation or...",13 000 people receive wildfires evacuation ord...,0,0,1
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,0,0,1


Stopwords 제거 및 Lemmatization

`the`, `is`, `a`, `in`과 같은 불용어는 문법적으로는 필요하지만 재난 여부를 분류하는 데는 상대적으로 정보량이 적다.  
--> NLTK의 stopwords를 사용하여 제거

`floods`, `flooded`, `flooding`처럼 형태가 다른 단어를 가능한 같은 기본형으로 통일하기 위해 WordNetLemmatizer를 적용  
`not`, `no`, `nor`와 같은 부정어는 문장의 의미를 바꿀 수 있으므로 불용어에서 제외하지 않고 보존

In [ ]:
import nltk
import contextlib
import io

# Colab 환경에서는 아래 다운로드가 필요할 수 있음
# 네트워크가 없는 환경에서는 오류 메시지를 숨기고 fallback을 사용함
for package in ['stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            nltk.download(package, quiet=True)
    except Exception:
        pass

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag

try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    # NLTK stopwords 다운로드가 실패할 경우를 대비한 fallback
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stop_words = set(ENGLISH_STOP_WORDS)

# 부정어는 의미가 중요하므로 제거하지 않음
stop_words = stop_words - {'no', 'nor', 'not'}

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'
    elif treebank_tag.startswith('V'):
        return 'v'
    elif treebank_tag.startswith('N'):
        return 'n'
    elif treebank_tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def remove_stopwords_and_lemmatize(text):
    tokens = str(text).split()

    # 불용어 제거
    tokens = [token for token in tokens if token not in stop_words]

    if len(tokens) == 0:
        return ''

    # 품사 태깅 후 lemmatization
    try:
        tagged_tokens = pos_tag(tokens)
        normalized_tokens = [
            lemmatizer.lemmatize(token, get_wordnet_pos(pos))
            for token, pos in tagged_tokens
        ]
    except Exception:
        # POS tagger나 wordnet 사용이 불가능한 경우 stemming으로 대체
        normalized_tokens = [stemmer.stem(token) for token in tokens]

    return ' '.join(normalized_tokens)

In [ ]:
train['text_model'] = train['text_clean'].apply(remove_stopwords_and_lemmatize)
test['text_model'] = test['text_clean'].apply(remove_stopwords_and_lemmatize)

train[['text_clean', 'text_model']].head()

,text_clean,text_model
0,our deeds are the reason of this earthquake ma...,deed reason earthquake may allah forgive u
1,forest fire near la ronge sask canada,forest fire near la ronge sask canada
2,all residents asked to shelter in place are be...,resident ask shelter place notify officer no e...
3,13 000 people receive wildfires evacuation ord...,13 000 people receive wildfire evacuation orde...
4,just got sent this photo from ruby alaska as s...,get sent photo ruby alaska smoke wildfires pou...


해시태그 보존 여부 확인

해시태그는 `#` 기호만 제거되고, 뒤의 단어는 `text_clean`과 `text_model`에 남아 있어야 한다.  
실제 해시태그가 포함된 tweet에서 처리 결과를 확인하는 과정:

In [ ]:
train.loc[
    train['text'].str.contains(r'#\w+', regex=True, na=False),
    ['text', 'text_clean', 'text_model']
].head()

,text,text_clean,text_model
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,deed reason earthquake may allah forgive u
3,"13,000 people receive #wildfires evacuation or...",13 000 people receive wildfires evacuation ord...,13 000 people receive wildfire evacuation orde...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,get sent photo ruby alaska smoke wildfires pou...
5,#RockyFire Update => California Hwy. 20 closed...,rockyfire update california hwy 20 closed in b...,rockyfire update california hwy 20 closed dire...
6,#flood #disaster Heavy rain causes flash flood...,flood disaster heavy rain causes flash floodin...,flood disaster heavy rain cause flash flood st...


keyword 정리

`nuclear%20reactor`처럼 URL encoding된 값은 `nuclear reactor`처럼 사람이 읽을 수 있는 형태로 변환

In [ ]:
def clean_keyword(keyword):
    keyword = urllib.parse.unquote(str(keyword))
    keyword = keyword.lower()
    keyword = re.sub(special_pattern, ' ', keyword)
    keyword = re.sub(space_pattern, ' ', keyword).strip()
    return keyword

train['keyword_clean'] = train['keyword'].apply(clean_keyword)
test['keyword_clean'] = test['keyword'].apply(clean_keyword)

train[['keyword', 'keyword_clean']].head()

,keyword,keyword_clean
0,unknown,unknown
1,unknown,unknown
2,unknown,unknown
3,unknown,unknown
4,unknown,unknown


location 정리

`location`은 사용자가 자유롭게 입력한 값이기 때문에 형식이 일정하지 않다.  --> 소문자 변환, URL/멘션 처리, 특수문자 제거, 공백 정리를 수행  
위치 정보는 결측치와 불규칙성이 많기 때문에 이번 단계에서는 주로 보조 변수로 정리

In [ ]:
def clean_location(location):
    location = html.unescape(str(location))
    location = location.lower()
    location = re.sub(url_pattern, ' urltoken ', location)
    location = re.sub(mention_pattern, ' usertoken ', location)
    location = location.encode('ascii', 'ignore').decode('ascii')
    location = re.sub(special_pattern, ' ', location)
    location = re.sub(space_pattern, ' ', location).strip()
    return location

train['location_clean'] = train['location'].apply(clean_location)
test['location_clean'] = test['location'].apply(clean_location)

train[['location', 'location_clean']].head()

,location,location_clean
0,unknown,unknown
1,unknown,unknown
2,unknown,unknown
3,unknown,unknown
4,unknown,unknown


keyword와 text 결합

최종 모델 입력값으로 `combined_text`를 생성  
`text_model`은 정제된 트윗 본문, `keyword_clean`은 트윗의 핵심 주제 정보  
 두 정보를 결합하면 본문 정보와 keyword 정보를 동시에 반영

In [ ]:
train['combined_text'] = (train['keyword_clean'] + ' ' + train['text_model']).str.replace(r'\s+', ' ', regex=True).str.strip()
test['combined_text'] = (test['keyword_clean'] + ' ' + test['text_model']).str.replace(r'\s+', ' ', regex=True).str.strip()

train[['keyword_clean', 'text_model', 'combined_text']].head()

,keyword_clean,text_model,combined_text
0,unknown,deed reason earthquake may allah forgive u,unknown deed reason earthquake may allah forgi...
1,unknown,forest fire near la ronge sask canada,unknown forest fire near la ronge sask canada
2,unknown,resident ask shelter place notify officer no e...,unknown resident ask shelter place notify offi...
3,unknown,13 000 people receive wildfire evacuation orde...,unknown 13 000 people receive wildfire evacuat...
4,unknown,get sent photo ruby alaska smoke wildfires pou...,unknown get sent photo ruby alaska smoke wildf...


추가 feature 생성

트윗 길이, 정제 후 길이, 단어 수는 재난 트윗과 비재난 트윗의 차이를 설명하는 보조 feature로 사용

In [ ]:
for df in [train, test]:
    df['text_length_raw'] = df['text'].astype(str).str.len()
    df['text_length_clean'] = df['text_clean'].str.len()
    df['word_count_clean'] = df['text_model'].apply(lambda x: len(x.split()) if x else 0)

train[['text_length_raw', 'text_length_clean', 'word_count_clean', 'has_url', 'has_mention', 'has_hashtag']].head()

,text_length_raw,text_length_clean,word_count_clean,has_url,has_mention,has_hashtag
0,69,68,7,0,0,1
1,38,37,7,0,0,0
2,133,130,12,0,0,0
3,65,63,8,0,0,1
4,88,85,9,0,0,1


전처리 결과 최종 확인

In [ ]:
train[[
    'text', 'text_clean', 'text_model',
    'keyword', 'keyword_clean',
    'location', 'location_clean',
    'combined_text'
]].head()

,text,text_clean,text_model,keyword,keyword_clean,location,location_clean,combined_text
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,deed reason earthquake may allah forgive u,unknown,unknown,unknown,unknown,unknown deed reason earthquake may allah forgi...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,forest fire near la ronge sask canada,unknown,unknown,unknown,unknown,unknown forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,resident ask shelter place notify officer no e...,unknown,unknown,unknown,unknown,unknown resident ask shelter place notify offi...
3,"13,000 people receive #wildfires evacuation or...",13 000 people receive wildfires evacuation ord...,13 000 people receive wildfire evacuation orde...,unknown,unknown,unknown,unknown,unknown 13 000 people receive wildfire evacuat...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,get sent photo ruby alaska smoke wildfires pou...,unknown,unknown,unknown,unknown,unknown get sent photo ruby alaska smoke wildf...


In [ ]:
test[[
    'text', 'text_clean', 'text_model',
    'keyword', 'keyword_clean',
    'location', 'location_clean',
    'combined_text'
]].head()

,text,text_clean,text_model,keyword,keyword_clean,location,location_clean,combined_text
0,Just happened a terrible car crash,just happened a terrible car crash,happen terrible car crash,unknown,unknown,unknown,unknown,unknown happen terrible car crash
1,"Heard about #earthquake is different cities, s...",heard about earthquake is different cities sta...,heard earthquake different city stay safe ever...,unknown,unknown,unknown,unknown,unknown heard earthquake different city stay s...
2,"there is a forest fire at spot pond, geese are...",there is a forest fire at spot pond geese are ...,forest fire spot pond geese flee across street...,unknown,unknown,unknown,unknown,unknown forest fire spot pond geese flee acros...
3,Apocalypse lighting. #Spokane #wildfires,apocalypse lighting spokane wildfires,apocalypse light spokane wildfire,unknown,unknown,unknown,unknown,unknown apocalypse light spokane wildfire
4,Typhoon Soudelor kills 28 in China and Taiwan,typhoon soudelor kills 28 in china and taiwan,typhoon soudelor kill 28 china taiwan,unknown,unknown,unknown,unknown,unknown typhoon soudelor kill 28 china taiwan


TF-IDF 입력 준비

전처리 단계 이후 모델링에 사용할 입력 변수는 `combined_text`로 설정  
실제 train/validation 분할을 할 때는 class 비율이 유지되도록 `stratify=train['target']`을 사용

In [ ]:
X = train['combined_text']
y = train['target']
X_test = test['combined_text']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_valid)
X_test_tfidf = tfidf.transform(X_test)

print('X_train_tfidf:', X_train_tfidf.shape)
print('X_valid_tfidf:', X_valid_tfidf.shape)
print('X_test_tfidf:', X_test_tfidf.shape)

X_train_tfidf: (6090, 10690)
X_valid_tfidf: (1523, 10690)
X_test_tfidf: (3263, 10690)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

val_pred = model.predict(X_valid_tfidf)
print("Validation Accuracy:", accuracy_score(y_valid, val_pred))

Validation Accuracy: 0.8082731451083388


# bert

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("현재 사용 device:", device)

현재 사용 device: cuda


In [ ]:
!pip install transformers datasets accelerate -U

BERT는 TF-IDF 벡터를 쓰지 않음. 대신 원문 텍스트를 BERT 전용 숫자로 변환해야 함. 가장 안정적인 bert-base-uncased 모델을 사용함.

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import f1_score

# 1. 메모리 및 캐시 완전 초기화
gc.collect()
torch.cuda.empty_cache()

# 2. 모델 선택 (가장 가벼운 DistilBERT)
model_nm = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_nm)

# 3. 데이터셋 샘플링 (속도를 위해 학습 데이터의 30%만 무작위 추출)
# 일단 돌아가는걸 확인하고, 시간이 남으면 전체 데이터로 늘리세요.
train_subset = train.loc[X_train.index].sample(frac=0.3, random_state=42)
train_texts = train_subset['combined_text'].values
train_labels = y_train.loc[train_subset.index].values

valid_texts = train['combined_text'].loc[X_valid.index].values
valid_labels = y_valid.values

# 4. 토크나이징
def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64) # 길이를 64로 줄여 속도 2배 향상

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
valid_ds = Dataset.from_dict({'text': valid_texts, 'label': valid_labels})

tokenized_train = train_ds.map(tokenize_func, batched=True)
tokenized_valid = valid_ds.map(tokenize_func, batched=True)

# 5. 모델 및 학습 설정
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, predictions)}

args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=32, # 배치 사이즈 업
    num_train_epochs=1,             # 딱 1번만 돌려서 결과 확인!
    fp16=True,                      # GPU 가속 필수
    report_to="none"
)

# 6. 실행
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,No log,0.434637,0.783019


TrainOutput(global_step=58, training_loss=0.5177476488310715, metrics={'train_runtime': 6.5707, 'train_samples_per_second': 278.052, 'train_steps_per_second': 8.827, 'total_flos': 30252242168064.0, 'train_loss': 0.5177476488310715, 'epoch': 1.0})

최종 bert

In [ ]:
import torch, gc
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import f1_score

# 1. 환경 정리 및 GPU 설정
gc.collect()
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

# 2. 데이터 준비 (전체 데이터 사용)
# 런타임 재시작으로 변수가 없다면 아래 주석을 풀고 다시 로드하세요.
# train = pd.read_csv('train.csv')
train_texts = train['combined_text'].loc[X_train.index].values
train_labels = y_train.values
valid_texts = train['combined_text'].loc[X_valid.index].values
valid_labels = y_valid.values

# 3. 모델 및 토크나이저 설정 (DistilBERT)
model_nm = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_nm)

def tokenize_func(examples):
    # max_length를 128로 복원하여 정보 손실 방지
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
valid_ds = Dataset.from_dict({'text': valid_texts, 'label': valid_labels})

tokenized_train = train_ds.map(tokenize_func, batched=True)
tokenized_valid = valid_ds.map(tokenize_func, batched=True)

# 4. 모델 로드 및 메트릭 정의
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, predictions)}

# 5. 고성능/세션안전 학습 설정
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",       # 평가 주기: 에포크마다
    save_strategy="epoch",       # 저장 주기: 에포크마다 (평가 주기와 일치시켜야 에러 안 남)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,                   # GPU 가속
    load_best_model_at_end=True, # 가장 성능 좋은 모델 최종 선택
    metric_for_best_model="f1",
    report_to="none",
    save_total_limit=1           # 모델 파일을 딱 1개만 남겨서 코랩 용량 방어
)

# 6. Trainer 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    compute_metrics=compute_metrics
)

print("학습을 시작합니다.")
trainer.train()

# 7. 최종 결과 확인
print("\n[최종 검증 데이터 성능]")
final_results = trainer.evaluate()
print(f"최종 F1 Score: {final_results['eval_f1']:.4f}")

현재 사용 중인 장치: cuda


Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[최종 검증 데이터 성능]


Training Loss,Validation Loss,Epoch,F1
0.329421,0.414619,3,0.805755


최종 F1 Score: 0.8058


# logistic regression

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install optuna
import optuna
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

optuna.logging.set_verbosity(optuna.logging.WARNING)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.1 MB/s eta 0:00:00


In [ ]:
# Logistic Regression Optuna 튜닝 함수
def tune_logistic_regression(
    X_train, y_train,
    X_eval, y_eval,
    n_trials=30,
    timeout=None,
    random_state=42
):
    def objective(trial):
        params = {
            'C': trial.suggest_float('C', 1e-3, 20.0, log=True),
            'solver': trial.suggest_categorical('solver', ['liblinear', 'lbfgs']),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
            'max_iter': 3000,
            'random_state': random_state,
            'n_jobs': -1,
        }

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        pred = model.predict(X_eval)
        return f1_score(y_eval, pred)

    study = optuna.create_study(
        direction='maximize',
        study_name='logistic_regression_f1'
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        timeout=timeout,
        show_progress_bar=True
    )

    best_params = study.best_params.copy()

    final_params = {
        **best_params,
        'max_iter': 3000,
        'random_state': random_state,
        'n_jobs': -1,
    }

    model = LogisticRegression(**final_params)
    model.fit(X_train, y_train)

    eval_pred = model.predict(X_eval)
    eval_f1 = f1_score(y_eval, eval_pred)

    return model, best_params, eval_f1, study

In [ ]:
# Optuna 실행
logistic_model, logistic_best_params, logistic_f1, logistic_study = tune_logistic_regression(
    X_train_tfidf,
    y_train,
    X_valid_tfidf,
    y_valid,
    n_trials=20,
    timeout=None,
    random_state=42
)

print('===== Logistic Regression Best Params =====')
print(logistic_best_params)
print(f'Validation F1 Score: {logistic_f1:.6f}')

  0%|          | 0/20 [00:00<?, ?it/s]

===== Logistic Regression Best Params =====
{'C': 0.9272243395514844, 'solver': 'liblinear', 'class_weight': 'balanced'}
Validation F1 Score: 0.770992


In [ ]:
# Threshold 최적화
def get_best_threshold(y_true, proba, start=0.10, end=0.90, step=0.001):
    thresholds = np.arange(start, end + step, step)
    f1_scores = [
        f1_score(y_true, (proba >= threshold).astype(int))
        for threshold in thresholds
    ]

    best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx]), float(f1_scores[best_idx])

In [ ]:
valid_proba = logistic_model.predict_proba(X_valid_tfidf)[:, 1]

best_threshold, best_f1 = get_best_threshold(
    y_valid,
    valid_proba,
    start=0.10,
    end=0.90,
    step=0.001
)

print('Best threshold:', best_threshold)
print('Best F1:', best_f1)

Best threshold: 0.4750000000000003
Best F1: 0.7718518518518519


In [ ]:
# Test 예측 및 submission 저장
test_proba = logistic_model.predict_proba(X_test_tfidf)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'target': test_pred
})

submission.to_csv('submission_logistic_regression.csv', index=False)

print('Saved submission_logistic_regression.csv')

Saved submission_logistic_regression.csv


# random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 1. 모델 정의 (하이퍼파라미터는 기본값에서 점진적으로 조정)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. 모델 학습
rf_model.fit(X_train_tfidf, y_train)

# 3. 검증 데이터 예측 및 평가
y_pred_rf = rf_model.predict(X_valid_tfidf)

print("Random Forest F1 Score:", f1_score(y_valid, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_valid, y_pred_rf))

Random Forest F1 Score: 0.7251975417032485

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.92      0.84       869
           1       0.85      0.63      0.73       654

    accuracy                           0.79      1523
   macro avg       0.81      0.77      0.78      1523
weighted avg       0.80      0.79      0.79      1523



In [ ]:
from scipy.sparse import hstack

# 학습에 사용할 보조 피처 리스트
meta_features = ['text_length_raw', 'text_length_clean', 'word_count_clean',
                 'has_url', 'has_mention', 'has_hashtag']

# 텍스트 데이터와 보조 피처 결합
X_train_combined = hstack([X_train_tfidf, train.loc[X_train.index, meta_features].values])
X_valid_combined = hstack([X_valid_tfidf, train.loc[X_valid.index, meta_features].values])
X_test_combined = hstack([X_test_tfidf, test[meta_features].values])

print(f"Combined Shape: {X_train_combined.shape}") # 피처 수가 늘어난 것을 확인

Combined Shape: (6090, 10696)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'], # 피처 수를 제한해서 속도 향상
    'class_weight': ['balanced']
}

random_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=10, # 딱 10번만 조합을 뽑아서 실행 (조절 가능)
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42
)

random_rf.fit(X_train_combined, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   param_distributions={'class_weight': ['balanced'],
                                        'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='f1', verbose=1)

In [ ]:
from sklearn.metrics import f1_score, classification_report

# 1. 최적의 모델 추출
best_rf = random_rf.best_estimator_

# 2. 검증 데이터 예측
y_pred = best_rf.predict(X_valid_combined)

# 3. 결과 출력
print(f"최적 하이퍼파라미터: {random_rf.best_params_}")
print(f"검증 데이터 F1 Score: {f1_score(y_valid, y_pred):.4f}")
print("\n[Classification Report]")
print(classification_report(y_valid, y_pred))

최적 하이퍼파라미터: {'n_estimators': 100, 'min_samples_split': 5, 'max_features': 'log2', 'max_depth': None, 'class_weight': 'balanced'}
검증 데이터 F1 Score: 0.7433

[Classification Report]
              precision    recall  f1-score   support

           0       0.78      0.92      0.84       869
           1       0.86      0.65      0.74       654

    accuracy                           0.81      1523
   macro avg       0.82      0.79      0.79      1523
weighted avg       0.82      0.81      0.80      1523



In [ ]:
# 최종 rf
import numpy as np

# 확률값 추출
y_probs = best_rf.predict_proba(X_valid_combined)[:, 1]

# 0.3부터 0.7까지 0.01 단위로 테스트
thresholds = np.arange(0.3, 0.7, 0.01)
f1_scores = [f1_score(y_valid, y_probs >= t) for t in thresholds]

# 최적의 임계값 찾기
max_f1 = max(f1_scores)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"최적 임계값(Threshold): {best_threshold:.2f}")
print(f"임계값 조정 후 최대 F1 Score: {max_f1:.4f}")

최적 임계값(Threshold): 0.42
임계값 조정 후 최대 F1 Score: 0.7604


# rf+lg+bert 앙상블

In [ ]:
# BERT validation / test 확률값 생성
import numpy as np
import pandas as pd
from scipy.special import softmax
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report

bert_valid_logits = trainer.predict(tokenized_valid).predictions
bert_valid_proba = softmax(bert_valid_logits, axis=1)[:, 1]

test_texts = test['combined_text'].values
test_ds = Dataset.from_dict({'text': test_texts})
tokenized_test = test_ds.map(tokenize_func, batched=True)

bert_test_logits = trainer.predict(tokenized_test).predictions
bert_test_proba = softmax(bert_test_logits, axis=1)[:, 1]

print("[BERT]")
print("valid proba shape:", bert_valid_proba.shape)
print("test  proba shape:", bert_test_proba.shape)

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

[BERT]
valid proba shape: (1523,)
test  proba shape: (3263,)


In [ ]:
# Logistic Regression validation / test 확률값 생성
lr_valid_proba = logistic_model.predict_proba(X_valid_tfidf)[:, 1]
lr_test_proba = logistic_model.predict_proba(X_test_tfidf)[:, 1]

print("\n[Logistic Regression]")
print("valid proba shape:", lr_valid_proba.shape)
print("test  proba shape:", lr_test_proba.shape)


[Logistic Regression]
valid proba shape: (1523,)
test  proba shape: (3263,)


In [ ]:
# Random Forest validation / test 확률값 생성
rf_valid_proba = best_rf.predict_proba(X_valid_combined)[:, 1]
rf_test_proba = best_rf.predict_proba(X_test_combined)[:, 1]

print("\n[Random Forest]")
print("valid proba shape:", rf_valid_proba.shape)
print("test  proba shape:", rf_test_proba.shape)


[Random Forest]
valid proba shape: (1523,)
test  proba shape: (3263,)


In [ ]:
# 개별 모델 성능 확인
def find_best_threshold(y_true, proba, start=0.10, end=0.90, step=0.001):
    thresholds = np.arange(start, end + step, step)

    scores = np.array([
        f1_score(y_true, (proba >= threshold).astype(int))
        for threshold in thresholds
    ])

    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])


model_probas = {
    'bert': bert_valid_proba,
    'logistic_regression': lr_valid_proba,
    'random_forest': rf_valid_proba
}

print("\n===== Individual Validation F1 =====")

individual_results = {}

for model_name, proba in model_probas.items():
    threshold, score = find_best_threshold(y_valid, proba)

    individual_results[model_name] = {
        'threshold': threshold,
        'f1': score
    }

    print(
        f"{model_name:20s} | "
        f"best threshold: {threshold:.3f} | "
        f"best F1: {score:.6f}"
    )


===== Individual Validation F1 =====
bert                 | best threshold: 0.695 | best F1: 0.810631
logistic_regression  | best threshold: 0.475 | best F1: 0.771852
random_forest        | best threshold: 0.428 | best F1: 0.760697


In [ ]:
# Ensemble weight + threshold 동시 탐색
# w_bert + w_lr + w_rf = 1이 되도록 탐색

weight_grid = np.arange(0.0, 1.01, 0.05)
threshold_grid = np.arange(0.10, 0.901, 0.001)

best_ensemble = {
    'f1': -1,
    'threshold': None,
    'w_bert': None,
    'w_lr': None,
    'w_rf': None
}

for w_bert in weight_grid:
    for w_lr in weight_grid:
        w_rf = 1.0 - w_bert - w_lr

        # weight 합이 1이 되지 않거나 음수 weight가 생기는 경우 제외
        if w_rf < -1e-9 or w_rf > 1:
            continue

        ensemble_valid_proba = (
            w_bert * bert_valid_proba +
            w_lr   * lr_valid_proba +
            w_rf   * rf_valid_proba
        )

        scores = np.array([
            f1_score(y_valid, (ensemble_valid_proba >= threshold).astype(int))
            for threshold in threshold_grid
        ])

        best_idx = int(np.argmax(scores))
        current_f1 = float(scores[best_idx])
        current_threshold = float(threshold_grid[best_idx])

        if current_f1 > best_ensemble['f1']:
            best_ensemble.update({
                'f1': current_f1,
                'threshold': current_threshold,
                'w_bert': float(w_bert),
                'w_lr': float(w_lr),
                'w_rf': float(w_rf)
            })

print("\n===== Best Ensemble Setting =====")
print(f"BERT weight                : {best_ensemble['w_bert']:.2f}")
print(f"Logistic Regression weight : {best_ensemble['w_lr']:.2f}")
print(f"Random Forest weight       : {best_ensemble['w_rf']:.2f}")
print(f"Best threshold             : {best_ensemble['threshold']:.3f}")
print(f"Validation F1              : {best_ensemble['f1']:.6f}")


===== Best Ensemble Setting =====
BERT weight                : 0.45
Logistic Regression weight : 0.20
Random Forest weight       : 0.35
Best threshold             : 0.579
Validation F1              : 0.814691


In [ ]:
# 최적 weight/threshold로 validation 결과 확인
ensemble_valid_proba = (
    best_ensemble['w_bert'] * bert_valid_proba +
    best_ensemble['w_lr']   * lr_valid_proba +
    best_ensemble['w_rf']   * rf_valid_proba
)

ensemble_valid_pred = (
    ensemble_valid_proba >= best_ensemble['threshold']
).astype(int)

print("\n===== Ensemble Validation Report =====")
print(classification_report(y_valid, ensemble_valid_pred))


===== Ensemble Validation Report =====
              precision    recall  f1-score   support

           0       0.83      0.94      0.88       869
           1       0.90      0.75      0.81       654

    accuracy                           0.85      1523
   macro avg       0.86      0.84      0.85      1523
weighted avg       0.86      0.85      0.85      1523



In [ ]:
# 최종 test 예측 및 submission 저장
ensemble_test_proba = (
    best_ensemble['w_bert'] * bert_test_proba +
    best_ensemble['w_lr']   * lr_test_proba +
    best_ensemble['w_rf']   * rf_test_proba
)

ensemble_test_pred = (
    ensemble_test_proba >= best_ensemble['threshold']
).astype(int)

submission_ensemble = pd.DataFrame({
    'id': test['id'],
    'target': ensemble_test_pred
})

submission_ensemble.to_csv(
    'submission_ensemble_bert_rf_logistic.csv',
    index=False
)

print("\nSaved: submission_ensemble_bert_rf_logistic.csv")
submission_ensemble.head()


Saved: submission_ensemble_bert_rf_logistic.csv


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


# Stacking

In [ ]:
# 스태킹에 사용할 bert 모델
bert_model = model

In [ ]:
# BERT Probability Function for Stacking
batch_size = 16
def get_bert_proba(texts, model, tokenizer, device, batch_size=batch_size):
    model.eval()
    all_probs = []
    ds = Dataset.from_dict({'text': list(texts)})
    ds = ds.map(lambda ex: tokenizer(ex['text'], padding='max_length',
                                      truncation=True, max_length=BERT_MAX_LEN),
                batched=True)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    loader = DataLoader(ds, batch_size=batch_size)
    with torch.no_grad():
        for batch in loader:
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device))
            probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs)

In [ ]:
# 베이스 모델 정의(최적 파라미터)
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

RANDOM_STATE = 42

lr_params = dict(
    C=2.006743556804092, solver='lbfgs', class_weight='balanced',
    max_iter=5000, random_state=RANDOM_STATE, n_jobs=-1,
)

rf_params = dict(
    n_estimators=100, max_depth=None, min_samples_split=5,
    max_features='log2', class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1,
)

xgb_params = dict(
    n_estimators=348, max_depth=4, learning_rate=0.16777676800210936,
    subsample=0.6996773747081134, colsample_bytree=0.7413710888156484,
    min_child_weight=2, gamma=0.07780132529155989,
    reg_alpha=0.00012886442636842542, reg_lambda=0.9313603443735947,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1,
)

meta_features = ['text_length_raw', 'text_length_clean',
                 'word_count_clean', 'has_url', 'has_mention', 'has_hashtag']

In [ ]:
# OOF 스태킹
from torch.utils.data import DataLoader

BERT_MAX_LEN  = 128

X_all_tfidf  = tfidf.transform(train['combined_text'])
train_reset  = train.reset_index(drop=True)
y_all        = train['target'].reset_index(drop=True).values
n_train      = X_all_tfidf.shape[0]
n_test       = X_test_tfidf.shape[0]
N_SPLITS     = 5

train_meta_sp = csr_matrix(train_reset[meta_features].values)
test_meta_sp  = csr_matrix(test[meta_features].values)
X_all_rf      = hstack([X_all_tfidf,  train_meta_sp])
X_test_rf     = hstack([X_test_tfidf, test_meta_sp])

train_texts_all = train_reset['combined_text'].values
test_texts_all  = test['combined_text'].values

oof_meta  = np.zeros((n_train, 4))   # LR / RF / XGB / BERT
test_meta = np.zeros((n_test,  4))
bert_test_acc = np.zeros(n_test)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, val_idx) in enumerate(kf.split(np.zeros(n_train), y_all)):
    print(f"\n{'='*50}  Fold {fold+1}/{N_SPLITS}  {'='*50}")

    y_tr  = y_all[tr_idx]
    y_val = y_all[val_idx]

    # ── LR ──────────────────────────────────────────────────────────
    print("  [1/4] LR 학습 중...")
    lr = LogisticRegression(**lr_params)
    lr.fit(X_all_tfidf[tr_idx], y_tr)
    oof_meta[val_idx, 0]  = lr.predict_proba(X_all_tfidf[val_idx])[:, 1]
    test_meta[:, 0]      += lr.predict_proba(X_test_tfidf)[:, 1] / N_SPLITS

    # ── RF ──────────────────────────────────────────────────────────
    print("  [2/4] RF 학습 중...")
    rf = RandomForestClassifier(**rf_params)
    rf.fit(X_all_rf[tr_idx], y_tr)
    oof_meta[val_idx, 1]  = rf.predict_proba(X_all_rf[val_idx])[:, 1]
    test_meta[:, 1]      += rf.predict_proba(X_test_rf)[:, 1] / N_SPLITS

    # ── XGBoost ─────────────────────────────────────────────────────
    print("  [3/4] XGBoost 학습 중...")
    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X_all_tfidf[tr_idx], y_tr)
    oof_meta[val_idx, 2]  = xgb.predict_proba(X_all_tfidf[val_idx])[:, 1]
    test_meta[:, 2]      += xgb.predict_proba(X_test_tfidf)[:, 1] / N_SPLITS

    # ── BERT (추론만) ────────────────────────────────────────────────
    print("  [4/4] BERT 추론 중...")
    oof_meta[val_idx, 3]  = get_bert_proba(train_texts_all[val_idx], bert_model, tokenizer, device)
    bert_test_acc        += get_bert_proba(test_texts_all, bert_model, tokenizer, device) / N_SPLITS

    gc.collect()
    torch.cuda.empty_cache()

test_meta[:, 3] = bert_test_acc
print("\n✅ OOF 스태킹 피처 생성 완료")
print(f"   oof_meta  shape: {oof_meta.shape}")
print(f"   test_meta shape: {test_meta.shape}")


==================================================  Fold 1/5  ==================================================
  [1/4] LR 학습 중...
  [2/4] RF 학습 중...
  [3/4] XGBoost 학습 중...
  [4/4] BERT 추론 중...


Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]


==================================================  Fold 2/5  ==================================================
  [1/4] LR 학습 중...
  [2/4] RF 학습 중...
  [3/4] XGBoost 학습 중...
  [4/4] BERT 추론 중...


Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]


==================================================  Fold 5/5  ==================================================
  [1/4] LR 학습 중...
  [2/4] RF 학습 중...
  [3/4] XGBoost 학습 중...
  [4/4] BERT 추론 중...


Map:   0%|          | 0/1522 [00:00<?, ? examples/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]


✅ OOF 스태킹 피처 생성 완료
   oof_meta  shape: (7613, 4)
   test_meta shape: (3263, 4)


In [ ]:
# 메타 러너 학습(Logistic Regression)
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
meta_lr.fit(oof_meta, y_all)

oof_proba = meta_lr.predict_proba(oof_meta)[:, 1]

best_thr, best_f1 = 0.5, 0.0
for thr in np.arange(0.30, 0.70, 0.001):
    score = f1_score(y_all, (oof_proba >= thr).astype(int))
    if score > best_f1:
        best_f1, best_thr = score, thr

print(f"\n[메타 러너 결과]")
print(f"  OOF 최적 Threshold : {best_thr:.3f}")
print(f"  OOF F1 Score       : {best_f1:.4f}")

feature_names = ['LR', 'RF', 'XGB', 'BERT']
print("\n  [각 모델 가중치 (메타 러너 계수)]")
for name, coef in zip(feature_names, meta_lr.coef_[0]):
    print(f"    {name:6s}: {coef:+.4f}")


[메타 러너 결과]
  OOF 최적 Threshold : 0.524
  OOF F1 Score       : 0.8726

  [각 모델 가중치 (메타 러너 계수)]
    LR    : -0.7796
    RF    : -0.0720
    XGB   : +0.0737
    BERT  : +6.3435


In [ ]:
# 최종 예측 및 submission 저장
test_proba = meta_lr.predict_proba(test_meta)[:, 1]
test_pred  = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({'id': test['id'], 'target': test_pred})
submission.to_csv('submission_stacking.csv', index=False)

print(f"\n✅ submission_stacking.csv 저장 완료")
print(f"   예측 분포: {pd.Series(test_pred).value_counts().to_dict()}")


✅ submission_stacking.csv 저장 완료
   예측 분포: {0: 2013, 1: 1250}
